In [ ]:
!pip install openai
# !pip install langchain_core==0.3.72
# !pip install langchain==0.3.27
!pip install langchain_community==0.3.31
# !pip install bitsandbytes
!pip install datasets
!pip install langgraph
!pip install langchain_google_genai==2.1.12
!pip install langchain_huggingface==0.3.1
# !pip install dppy
!pip install vllm==0.8.2
# !pip install llama-cpp-python

In [ ]:
allowed_llm_sources = ['google', 'gguf-llama-cpp', 'gguf-ctransformers' , 'transformers-pipeline']
llm_source = "transformers-pipeline"
# fill this part if llm has a gguf file
gguf_path = "models/gguf/llama-2-7b.Q4_K_M.gguf"
embedding_sources = ['google', 'huggingface']
embedding_source = embedding_sources[1]

In [ ]:
import os
import re
from sklearn.utils import shuffle
import pandas as pd
import numpy as np
import random
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.optim import Adam
from transformers import get_scheduler
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
import spacy
from langchain.schema import Document
# from dppy.finite_dpps import FiniteDPP
import collections
import json
def lists_are_equal(list1,list2):
    if collections.Counter(list1) == collections.Counter(list2):
        return True
    else:
        return False

def find_most_frequent_list(lists):
    equal_counters = []
    for i in range(len(lists)):
        counter = 0
        for j in range(len(lists)):
            if lists_are_equal(lists[i],lists[j]):
                counter += 1
        equal_counters.append(counter);
    return lists[np.argmax(np.array(equal_counters))]

# Function to generate text
def generate_text(tokenizer, model, prompt, max_length=10000, temperature=0.1, top_p=0.9, skip_prompt=True):
    device = "cuda" if torch.cuda.is_available() else "cpu"  # Use GPU if available
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_token_length = inputs["input_ids"].shape[-1]  # Get the number of tokens in the prompt
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    # Decode the output, optionally skipping the prompt
    decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    if skip_prompt:
        # Convert tokens to text, starting after the prompt tokens
        generated_tokens = outputs[0][prompt_token_length:]
        decoded_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return decoded_output
class T5ProjNetRetrievalEvaluator():

    def __init__(self):
        self.seed = 42
        self.batch_size = 1
        self.num_epochs = 50
        self.LOW_LABEL = 0
        self.AMBIGOUS_LABEL = 1
        self.HIGH_LABEL = 2
        self.nlp = spacy.load('en_core_web_sm')
        self.nlp.max_length = 10000000
        # self.s_model= SentenceTransformer('sentence-transformers/all-roberta-large-v1').to('cpu')
        # self.s_model= SentenceTransformer('sentence-transformers/all-roberta-large-v1')
        # self.s_model= SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-cos-v1')
        # self.s_model= SentenceTransformer('sentence-transformers/multi-qa-mpnet-base-dot-v1')
        self.number_of_selected = {
            "highs": [],
            "mediums": [],
            "websearches": []
        }

    def load_pretrained_model(self, load_path='./outputs/ep8'):
        # Define paths and parameters
        base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"  # Base TinyLlama model
        lora_weights_path = "fine-tuned-models/qwen_lora_finetuned_batch8_one_sentence_reasoning_correctformat_newprompt"
        offload_dir = "offload_dir"  # Directory for offloading weights to disk
        device = "cuda" if torch.cuda.is_available() else "cpu"  # Use GPU if available

        # Create offload directory if it doesn't exist
        os.makedirs(offload_dir, exist_ok=True)

        # Load the tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)

        # Load the base model with offloading support
        try:
            self.base_model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                torch_dtype=torch.float16 if device == "cuda" else torch.float32,
                device_map="auto",
                offload_folder=offload_dir,  # Specify offload directory
                low_cpu_mem_usage=True  # Optimize memory usage during loading
            )
        except RuntimeError as e:
            print(f"Error loading model: {e}")
            print("Falling back to CPU-only mode...")
            self.base_model = AutoModelForCausalLM.from_pretrained(
                base_model_name,
                torch_dtype=torch.float32,
                device_map={"": "cpu"},  # Force CPU placement
                offload_folder=offload_dir
            )

        # Load the LoRA weights onto the base model
        self.model = PeftModel.from_pretrained(
            self.base_model,
            lora_weights_path,
            device_map="auto" if device == "cuda" else {"": "cpu"},
            offload_folder=offload_dir
        )

        # Set model to evaluation mode
        self.model.eval()
        return self

    def evaluate_single(self, q: str, doc: str, mode='passage') -> float: # or strip
        # we can use few shot to show mid levels
        if not (type(doc) is str):
          doc = doc.page_content
        # prompt_template = "Question: {question} Context: {context} \n\nTo what degree can we confidently assert that the context provides a clear and accurate answer to the question? Answer in *JSON* with two keys: rating (one of \"low\", \"medium\", or \"high\") and reason (a sentence explaining your judgment)."
        # pub_prompt_template = "Question: Is the following statement correct? {question} Context: {context} \n\nTo what degree can we confidently assert that the context provides a clear and accurate answer to the question?  Answer in *JSON* with two keys: rating (one of \"low\", \"medium\", or \"high\") and reason (a sentence explaining your judgment)."
        prompt = f"Question: {q} Context: {doc} \n\nTo what degree can we confidently assert that the context provides a clear and accurate answer to the question? Answer in *JSON* with two keys: rating (one of \"low\", \"medium\", or \"high\") and reason (a sentence explaining your judgment). \n\nAnswer: "
        response = generate_text(self.tokenizer, self.model ,prompt, skip_prompt=True)
        # print(response.strip())
        if('"rating": "high"' in response.strip().lower()):
          try:
            result = json.loads(response.strip())
            # print(result)
            return 1.0,2, Document(page_content=str(result['reason']), metadata={"source": "Knowledge"}) if 'reason' in result.keys() else doc
          except:
            return 1.0,2, doc
        if('"rating": "medium"' in response.strip().lower()):
          try:
            result = json.loads(response.strip())
            # print(result)
            return 1.0,1, Document(page_content=str(result['reason']), metadata={"source": "Knowledge"}) if 'reason' in result.keys() else doc
          except:
            return 1.0,1, doc
        else:
          try:
            result = json.loads(response.strip())
            # print(result)
            return 1.0,0, Document(page_content=str(result['reason']), metadata={"source": "Knowledge"}) if 'reason' in result.keys() else doc
          except:
            return 1.0,0, doc

    def evaluate_batch(self, q: str, docs: list):
        # Initialize lists for different categories
        high_probs = []
        medium_probs = []
        low_probs = []
        high_docs = []
        medium_docs = []
        low_docs = []

        # Sort strips using sort_docs
        sorted_docs, _ = self.sort_docs(q, docs, docs)

        # Evaluate sorted strips and categorize, break if high category reaches 3
        for doc in sorted_docs:
            prob, out, text = self.evaluate_single(q, doc)
            if out == self.HIGH_LABEL:
                high_docs.append(text)
                high_probs.append(prob)
            elif out == self.AMBIGOUS_LABEL:
                medium_docs.append(text)
                medium_probs.append(prob)
            else:
                low_docs.append(text)
                low_probs.append(prob)

        # Check conditions for different outcomes
        if len(high_docs) > 0:  # At least one strip is labeled "high"
            self.number_of_selected["highs"].append(len(high_docs) if len(high_docs) < 3 else 3)
            self.number_of_selected["mediums"].append(0)
            self.number_of_selected["websearches"].append(0)
            return high_docs[:3], "No"
        elif len(medium_docs) > 0:  # Ambiguous condition
            self.number_of_selected["highs"].append(0)
            self.number_of_selected["mediums"].append(len(medium_docs) if len(medium_docs) < 3 else 3)
            self.number_of_selected["websearches"].append(0)
            return medium_docs[:3], "Yes"
        else:
            self.number_of_selected["highs"].append(0)
            self.number_of_selected["mediums"].append(0)
            self.number_of_selected["websearches"].append(0)
            return [], "Yes"
    def evaluate_websearch(self, q: str, docs: list, num_of_medium_docs=0,i=0):
          # convert docs to strips. this is knowledge refinement
          new_docs, _ = self.sort_docs(q,docs,docs)
          new_docs = docs[:5]
          high_docs = []
          high_probs = []
          for d in new_docs:
          # # for d in strips:
              prob, out, text = self.evaluate_single(q, str(d))
              if out == self.HIGH_LABEL:
                  high_docs.append(str(text))
                  high_probs.append(prob)
          self.number_of_selected["websearches"].append(len(high_docs) if len(high_docs) < 3 else 3)
          return high_docs[:5 if num_of_medium_docs == 0 else 3], "No"

In [ ]:
import pickle

In [ ]:
from src.DataLoaders.DataLoader import DataLoader
from src.DataLoaders.Arc import Arc
from src.DataLoaders.PubHealth import PubHealth
from src.DataLoaders.PopQA import PopQA
from src.RetrievalEvaluators.RetrievalEvaluator import RetrievalEvaluator
from graph import workflow_compiler
app = workflow_compiler()
# dataloader, retrieval_evaluator= Arc(), T5ProjNetRetrievalEvaluator().load_pretrained_model() # for ARC
# dataloader, retrieval_evaluator= PubHealth(), T5ProjNetRetrievalEvaluator().load_pretrained_model() # for Pubhealth
dataloader, retrieval_evaluator= PopQA(), T5ProjNetRetrievalEvaluator().load_pretrained_model() # for PopQA
# from models.LLM import llm

In [ ]:
from tqdm import tqdm
dataloader.load_data()
with open('/KD/context/popqa-rationales.pickle', 'rb') as file:
    final_contexts = pickle.load(file)
# final_contexts = []
def save_outputs(save_path: str = '/KD/context/popqa-rationales.pickle') -> bool:
     with open(save_path, 'wb') as handle:
         pickle.dump(final_contexts, handle)
questions = dataloader.input_test_data
for i,c in tqdm(enumerate(dataloader.contexts)):
    if i < len(final_contexts):
        continue
    final_contexts.append([])
    for j,doc in enumerate(c):
      final_contexts[i].append(retrieval_evaluator.evaluate_single(questions[i],doc))
    save_outputs()

1399it [1:27:39,  3.76s/it]


In [ ]:
for i,c in enumerate(final_contexts):
  if len(c) == 0:
    print(i)

In [ ]:
len(final_contexts)

1399